In [7]:
"""
Apriori Algorithm and Association Rule Mining

This module implements the Apriori algorithm for mining frequent itemsets and generating association rules from transactional datasets.
It provides a clear, modular, and extensible implementation suitable for educational, research, or practical use on small to medium datasets.

Objective:
-----------
The main objective of this code is to identify frequent itemsets in a collection of transactions and to extract meaningful association rules
that describe relationships between items. The code supports flexible support, confidence, and lift thresholds, and includes a demonstration
with a classic market-basket dataset.

Main Components:
----------------
- powerset: Generates all non-empty proper subsets of an itemset, used for rule generation.
- normalize_support_threshold: Converts minimum support to both absolute count and fractional form.
- count_itemsets: Counts occurrences of candidate itemsets in the transaction database.
- generate_L1: Finds frequent 1-itemsets.
- join_step: Produces candidate (k+1)-itemsets by joining frequent k-itemsets.
- has_infrequent_subset / prune_step: Prunes candidates with infrequent subsets (Apriori property).
- apriori: Main loop for mining all frequent itemsets.
- generate_association_rules: Generates association rules from frequent itemsets, computes support, confidence, lift, and interest, and filters/sorts rules.
- demo_transactions / run_demo: Provides a sample dataset and prints frequent itemsets and rules for demonstration.

Usage:
------
The module can be run as a script to see example output, or imported and used programmatically via the `apriori` and `generate_association_rules` functions.

"""

from __future__ import annotations
from collections import defaultdict
from itertools import combinations, chain
from typing import Dict, Iterable, List, Set, Tuple, FrozenSet, Any, Optional


# =========================
# Utilities
# =========================


def powerset(s: Iterable[Any]) -> Iterable[Tuple[Any, ...]]:
    """
    Return all non-empty proper subsets of iterable s (as tuples).
    Example: {a,b,c} -> (a,), (b,), (c,), (a,b), (a,c), (b,c)
    """
    s = tuple(s)
    for r in range(1, len(s)):
        for comb in combinations(s, r):
            yield comb


def normalize_support_threshold(
    min_support: float, n_transactions: int
) -> Tuple[int, float]:
    """
    If min_support >= 1 it's treated as an absolute count.
    If 0 < min_support < 1 it's treated as a fraction and converted to count.
    Returns (min_count, min_support_fraction)
    """
    if min_support >= 1:
        min_count = int(min_support)
        min_frac = min_support / n_transactions
    else:
        min_frac = float(min_support)
        min_count = int(min_support * n_transactions + 1e-12)  # floor
        # ensure at least 1 if min_support > 0
        if 0 < min_support < 1 and min_count == 0:
            min_count = 1
    return min_count, min_frac


# =========================
# Candidate generation & pruning
# =========================


def count_itemsets(
    transactions: List[Set[Any]], candidates: Set[FrozenSet[Any]]
) -> Dict[FrozenSet[Any], int]:
    """
    Count the number of transactions containing each candidate itemset.

    Args:
        transactions: List of transactions (each transaction is a set of items).
        candidates: Set of candidate itemsets (as frozensets).

    Returns:
        Dictionary mapping each candidate itemset to its support count.
    """
    counts = defaultdict(int)

    for transaction in transactions:
        for candidate in candidates:
            if candidate.issubset(transaction):
                counts[candidate] += 1
        
    return dict(counts)


def generate_L1(
    transactions: List[Set[Any]], min_count: int
) -> Dict[FrozenSet[Any], int]:
    """
    Generate frequent 1-itemsets (L1) and their support counts.

    Args:
        transactions: List of transactions (each transaction is a set of items).
        min_count: Minimum support count threshold.

    Returns:
        Dictionary mapping each frequent 1-itemset (as frozenset) to its support count.
    """
    counts = defaultdict(int)

    for transaction in transactions:
        for item in transaction:
            counts[frozenset([item])] += 1

    
    return {k: v for k, v in counts.items() if v >= min_count}


def join_step(Lk: Set[FrozenSet[Any]]) -> Set[FrozenSet[Any]]:
    """
    Generate candidate (k+1)-itemsets by joining pairs of frequent k-itemsets
    that share the first k-1 items (Apriori join step).

    Args:
        Lk: Set of frequent k-itemsets (as frozensets).

    Returns:
        Set of candidate (k+1)-itemsets (as frozensets).
    """
    Lk_list = sorted([tuple(sorted(x)) for x in Lk])
    k = len(Lk_list[0]) if Lk_list else 0
    Ck1 = set()
    for i in range(len(Lk_list)):
        for j in range(i + 1, len(Lk_list)):
            a, b = Lk_list[i], Lk_list[j]
            if a[: k - 1] == b[: k - 1]:  # share first k-1 items
                candidate = frozenset(a) | frozenset(b)
                if len(candidate) == k + 1:
                    Ck1.add(candidate)
            else:
                break  # because list is sorted; no more matches for this i
    return Ck1


def has_infrequent_subset(candidate: FrozenSet[Any], Lk: Set[FrozenSet[Any]]) -> bool:
    """
    Checks if a candidate itemset contains any (k)-subset that is not present in the set of frequent itemsets Lk.

    This function is typically used in the Apriori algorithm to prune candidate itemsets that have infrequent subsets.

    Parameters:
        candidate (FrozenSet[Any]): The candidate itemset to be checked.
        Lk (Set[FrozenSet[Any]]): The set of frequent itemsets of size k.

    Returns:
        bool: True if the candidate has any (k)-subset not in Lk (i.e., an infrequent subset), False otherwise.
    """
    k = len(candidate) - 1
    for subset in combinations(candidate, k):
        if frozenset(subset) not in Lk:
            return True
    return False


def prune_step(
    Ck1: Set[FrozenSet[Any]], Lk: Set[FrozenSet[Any]]
) -> Set[FrozenSet[Any]]:
    """
    Apply the Apriori prune: remove candidates that have an infrequent subset.
    """
    pruned = {c for c in Ck1 if not has_infrequent_subset(c, Lk)}
    return pruned


# =========================
# Apriori main
# =========================


def apriori(
    transactions: List[Set[Any]], min_support: float = 0.5
) -> Tuple[Dict[int, Dict[FrozenSet[Any], float]], Dict[FrozenSet[Any], float]]:
    """
    Run the Apriori algorithm to find frequent itemsets.

    Parameters:
        transactions (List[Set[Any]]):
            A list of transactions, where each transaction is represented as a set of items.
        min_support (float, optional):
            The minimum support threshold. If >= 1, it is treated as an absolute count of transactions.
            If 0 < min_support < 1, it is treated as a fraction of the total number of transactions.

    Returns:
        Tuple[
            Dict[int, Dict[FrozenSet[Any], float]],
            Dict[FrozenSet[Any], float]
        ]:
            - L_levels: A dictionary mapping k (itemset size) to a dictionary of frequent itemsets of size k.
              Each inner dictionary maps a frozenset of items to its support as a fraction of total transactions.
            - support_map: A dictionary mapping each frequent itemset (as a frozenset) to its support (fraction of transactions).
    """
    n = len(transactions)
    min_count, _ = normalize_support_threshold(min_support, n)

    # L1
    L1_counts = generate_L1(transactions, min_count)
    L_levels: Dict[int, Dict[FrozenSet[Any], float]] = {}
    support_map: Dict[FrozenSet[Any], float] = {}
    k = 1

    if not L1_counts:
        return L_levels, support_map

    Lk = set(L1_counts.keys())
    L_levels[k] = {s: c / n for s, c in L1_counts.items()}
    support_map.update(L_levels[k])

    # Iteratively generate L_{k+1}

    while Lk: 
        Ck = join_step(Lk)
        Ck = prune_step(Ck, Lk)

        if not Ck:
            break

        counts = count_itemsets(transactions, Ck)

        Lk_counts = {}
        for itemset, count in counts.items():
            if count >= min_count:
                Lk_counts[itemset] = count 

        if not Lk_counts:
            break

        k += 1
        Lk = set(Lk_counts.keys())
        L_levels[k] = {s: c / n for s, c in Lk_counts.items()}
        support_map.update(L_levels[k])

    return L_levels, support_map


# =========================
# Association rules
# =========================


def generate_association_rules(
    support_map: Dict[FrozenSet[Any], float],
    min_confidence: float = 0.6,
    min_lift: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """
    Generate association rules X -> Y from frequent itemsets in support_map.

    Parameters:
        support_map (Dict[FrozenSet[Any], float]):
            A dictionary mapping each frequent itemset (as a frozenset of items) to its support (as a fraction of total transactions).
        min_confidence (float, optional):
            The minimum confidence threshold for a rule to be included. Confidence is defined as support(X ∪ Y) / support(X).
            Default is 0.6.
        min_lift (Optional[float], optional):
            If provided, only rules with lift >= min_lift are included. Lift is defined as confidence / support(Y).
            Default is None (no lift filtering).

    Returns:
        List[Dict[str, Any]]:
            A list of association rules, where each rule is represented as a dictionary with the following keys:
                - 'antecedent': tuple of items (the X in X -> Y)
                - 'consequent': tuple of items (the Y in X -> Y)
                - 'support': float, support of X ∪ Y (fraction of transactions)
                - 'confidence': float, confidence of the rule (support(X ∪ Y) / support(X))
                - 'lift': float, lift of the rule (confidence / support(Y))
                - 'interest': float, interest measure of the rule (confidence - support(Y))
            The list is sorted in descending order by confidence, then lift, then support.
    """
    rules = []
    # Only consider itemsets of size >= 2 to form rules
    frequent_sets = [s for s in support_map if len(s) >= 2]

    for itemset in frequent_sets:
        s_xy = support_map[itemset]
        for A_tuple in powerset(itemset):
            A = frozenset(A_tuple)
            B = itemset - A
            if not B:
                continue
            s_x = support_map.get(A)
            s_y = support_map.get(B)
            if s_x is None or s_y is None or s_x == 0:
                continue
            confidence = s_xy / s_x
            lift = confidence / s_y
            interest = confidence - s_y
            if confidence >= min_confidence and (min_lift is None or lift >= min_lift):
                rules.append(
                    {
                        "antecedent": tuple(sorted(A)),
                        "consequent": tuple(sorted(B)),
                        "support": s_xy,
                        "confidence": confidence,
                        "lift": lift,
                        "interest": interest,
                    }
                )
    # Sort rules by (confidence desc, lift desc, support desc)
    rules.sort(key=lambda r: (r["confidence"], r["lift"], r["support"]), reverse=True)
    return rules


# =========================
# Example usage / demo
# =========================


def demo_transactions() -> List[Set[str]]:
    """
    Small illustrative dataset (classic market-basket style).
    """
    baskets = [
        {"milk", "bread", "eggs"},
        {"milk", "bread"},
        {"milk", "diapers", "beer", "chips"},
        {"bread", "diapers", "beer"},
        {"milk", "bread", "diapers", "beer"},
        {"bread", "eggs"},
        {"milk", "eggs"},
        {"diapers", "beer"},
        {"milk", "bread", "diapers"},
        {"milk", "bread", "beer"},
        {"milk", "bread", "butter"},
        {"bread", "cheese"},
        {"milk", "cereal"},
        {"bread", "eggs", "butter"},
        {"milk", "bread", "eggs", "cheese"},
        {"diapers", "beer", "chips"},
        {"milk", "bread", "cereal"},
        {"bread", "butter"},
        {"milk", "eggs", "cereal"},
        {"bread", "eggs", "cheese"},
    ]
    return [set(b) for b in baskets]


def run_demo(
    min_support: float = 0.3, min_confidence: float = 0.7, min_lift: float = 1.0
) -> None:
    """
    Run Apriori + rule mining on the demo dataset and print results.
    """
    transactions = demo_transactions()
    print(f"#transactions = {len(transactions)}")
    print(
        f"min_support = {min_support} ({'count' if min_support >= 1 else 'fraction'})"
    )
    print(f"min_confidence = {min_confidence}")
    if min_lift is not None:
        print(f"min_lift = {min_lift}")

    L_levels, support_map = apriori(transactions, min_support=min_support)

    print("\n=== Frequent Itemsets ===")
    for k in sorted(L_levels):
        print(f"L{k} (size {k} itemsets):")
        for itemset, supp in sorted(
            L_levels[k].items(), key=lambda x: (len(x[0]), x[1], tuple(sorted(x[0])))
        ):
            print(f"  {tuple(sorted(itemset))} -> support={supp:.3f}")

    print("\n=== Association Rules ===")
    rules = generate_association_rules(
        support_map, min_confidence=min_confidence, min_lift=min_lift
    )
    if not rules:
        print("  (none)")
    for r in rules:
        A = ", ".join(r["antecedent"])
        B = ", ".join(r["consequent"])
        print(
            f"  {{{A}}} -> {{{B}}} | supp={r['support']:.3f}, conf={r['confidence']:.3f}, lift={r['lift']:.3f}, interest={r['interest']:.3f}"
        )

if __name__ == "__main__":
    # Try different thresholds to see pruning & rules change
    run_demo(min_support=0.2, min_confidence=0.5, min_lift=1.0)


#transactions = 20
min_support = 0.2 (fraction)
min_confidence = 0.5
min_lift = 1.0

=== Frequent Itemsets ===
L1 (size 1 itemsets):
  ('beer',) -> support=0.300
  ('diapers',) -> support=0.300
  ('eggs',) -> support=0.350
  ('milk',) -> support=0.600
  ('bread',) -> support=0.700
L2 (size 2 itemsets):
  ('eggs', 'milk') -> support=0.200
  ('beer', 'diapers') -> support=0.250
  ('bread', 'eggs') -> support=0.250
  ('bread', 'milk') -> support=0.400

=== Association Rules ===
  {beer} -> {diapers} | supp=0.250, conf=0.833, lift=2.778, interest=0.533
  {diapers} -> {beer} | supp=0.250, conf=0.833, lift=2.778, interest=0.533
  {eggs} -> {bread} | supp=0.250, conf=0.714, lift=1.020, interest=0.014
